# BQL Export — MacroDesk Data

Exporta fundamentais, IV de opções, GEX SPX e LETF flows para CSVs.
O MacroDesk detecta automaticamente e atualiza o dashboard.

**Rodar célula 1** (config) → **Célula 2** (loop automático)

Loop exporta a cada **3 minutos**. Para parar: Kernel → Interrupt.

In [ ]:
# ── Célula 1: Configuração e funções ─────────────────────────────────────────
import bql
import csv
import math
import time
from datetime import date, datetime, timedelta
from pathlib import Path

bq = bql.Service()

# Destino dos CSVs — lido pelo MacroDesk
OUT = Path(r'C:\Users\rafael bentes\agente-workspace\bql_data')
OUT.mkdir(parents=True, exist_ok=True)

# Top 20 S&P 500 por peso (abr/2026) + referências
STOCKS = [
    'AAPL US Equity', 'MSFT US Equity', 'NVDA US Equity', 'AMZN US Equity', 'META US Equity',
    'GOOGL US Equity', 'TSLA US Equity', 'BRK/B US Equity', 'AVGO US Equity', 'JPM US Equity',
    'LLY US Equity', 'UNH US Equity', 'XOM US Equity', 'COST US Equity', 'V US Equity',
    'MA US Equity', 'WMT US Equity', 'NFLX US Equity', 'JNJ US Equity', 'PG US Equity',
    'SPY US Equity', 'QQQ US Equity', 'GLD US Equity', 'TLT US Equity', 'HYG US Equity',
]
TICKER = {s: s.replace(' US Equity', '').replace('BRK/B', 'BRK-B') for s in STOCKS}

def _v(row, col):
    """Valor float seguro — None se NaN."""
    try:
        v = float(row[col])
        return None if math.isnan(v) else v
    except Exception:
        return None

def _norm(df):
    """Remove parênteses dos nomes de coluna do BQL."""
    df.columns = [c.split('(')[0].strip() for c in df.columns]
    return df

def _csv(name, rows, fields):
    p = OUT / name
    with open(p, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        w.writeheader(); w.writerows(rows)
    print(f'  {name}: {len(rows)} linhas')

print('✓ BQL conectado. Pronto para exportar.')

In [ ]:
# ── Célula 2: Loop automático — exporta a cada 3 minutos ─────────────────────
# Kernel → Interrupt para parar

INTERVAL = 180  # segundos

def export_all():
    tstr = ', '.join(f'"{s}"' for s in STOCKS)
    eq   = ', '.join(f'"{s}"' for s in STOCKS if 'Equity' in s)

    # ── 1. Fundamentais ──────────────────────────────────────────────────────
    try:
        df = _norm(bql.combined_df(bq.execute(
            f'get(PE_RATIO, CUR_MKT_CAP, BETA, PROF_MARGIN,'
            f'TOT_DEBT_TO_TOT_EQY, RETURN_COM_EQY, EQY_DVD_YLD_IND, PX_LAST)'
            f' for([{tstr}])'
        )).groupby(level=0).last())
        df2 = _norm(bql.combined_df(bq.execute(
            f'get(PX_HIGH(dates=range(-365D,0D),frq=Y),'
            f'PX_LOW(dates=range(-365D,0D),frq=Y), PX_TO_BOOK_RATIO, EV_TO_T12M_EBITDA)'
            f' for([{tstr}])'
        )).groupby(level=0).last())
        rows = []
        for s in STOCKS:
            t  = TICKER[s]
            r1 = df.loc[s]  if s in df.index  else {}
            r2 = df2.loc[s] if s in df2.index else {}
            px = _v(r1, 'PX_LAST'); hi = _v(r2, 'PX_HIGH')
            mc = _v(r1, 'CUR_MKT_CAP')
            rows.append({
                'ticker': t,
                'pe':            round(_v(r1,'PE_RATIO'), 2)              if _v(r1,'PE_RATIO') else '',
                'pb':            round(_v(r2,'PX_TO_BOOK_RATIO'), 2)      if _v(r2,'PX_TO_BOOK_RATIO') else '',
                'ev_ebitda':     round(_v(r2,'EV_TO_T12M_EBITDA'), 2)     if _v(r2,'EV_TO_T12M_EBITDA') else '',
                'beta':          round(_v(r1,'BETA'), 3)                  if _v(r1,'BETA') else '',
                'mktcap_b':      round(mc / 1e9, 2)                       if mc else '',
                'roe':           round(_v(r1,'RETURN_COM_EQY') / 100, 4)  if _v(r1,'RETURN_COM_EQY') else '',
                'profit_margin': round(_v(r1,'PROF_MARGIN') / 100, 4)     if _v(r1,'PROF_MARGIN') else '',
                'debt_equity':   round(_v(r1,'TOT_DEBT_TO_TOT_EQY'), 2)   if _v(r1,'TOT_DEBT_TO_TOT_EQY') else '',
                'dividend_yield':round(_v(r1,'EQY_DVD_YLD_IND') / 100, 4) if _v(r1,'EQY_DVD_YLD_IND') else '',
                'price':         round(px, 4)                             if px else '',
                'hi_52w':        round(hi, 4)                             if hi else '',
                'lo_52w':        round(_v(r2,'PX_LOW'), 4)                if _v(r2,'PX_LOW') else '',
                'drawdown_52w':  round((px - hi) / hi, 4)                 if px and hi and hi > 0 else '',
            })
        _csv('fundamentals.csv', rows,
             ['ticker','pe','pb','ev_ebitda','beta','mktcap_b',
              'roe','profit_margin','debt_equity','dividend_yield',
              'price','hi_52w','lo_52w','drawdown_52w'])
    except Exception as e:
        print(f'  [ERRO] fundamentals: {e}')

    # ── 2. Options IV / Skew / PCR ───────────────────────────────────────────
    try:
        eq_list = [s for s in STOCKS if 'Equity' in s]
        r3 = bql.combined_df(bq.execute(bql.Request(eq_list, {
            'atm_iv': bq.data.implied_volatility(expiry='30D', pct_moneyness='100'),
            'put25':  bq.data.implied_volatility(expiry='30D', delta='25', put_call='PUT'),
            'call25': bq.data.implied_volatility(expiry='30D', delta='25', put_call='CALL'),
        }))).groupby(level=0).last()
        try:
            r4 = bql.combined_df(bq.execute(bql.Request(
                eq_list, bq.data.put_call_open_interest_ratio()
            ))).groupby(level=0).last()
            pcr_col = r4.columns[0]
        except Exception:
            r4 = None; pcr_col = None
        rows = []
        for s in eq_list:
            t = TICKER[s]
            row = r3.loc[s] if s in r3.index else {}
            va = _v(row, 'atm_iv'); vp = _v(row, 'put25'); vc = _v(row, 'call25')
            pcr = _v(r4.loc[s], pcr_col) if (r4 is not None and s in r4.index and pcr_col) else None
            rows.append({
                'ticker':  t,
                'atm_iv':  round(va / 100, 4)            if va else '',
                'skew_25d':round((vp - vc) / 100, 4)     if vp and vc else '',
                'pcr_oi':  round(pcr, 3)                 if pcr else '',
            })
        _csv('options_iv.csv', rows, ['ticker','atm_iv','skew_25d','pcr_oi'])
    except Exception as e:
        print(f'  [ERRO] options_iv: {e}')

    # ── 3. GEX SPX ───────────────────────────────────────────────────────────
    try:
        spot_df = _norm(bql.combined_df(bq.execute('get(PX_LAST) for(["SPX Index"])')).groupby(level=0).last())
        spot = float(spot_df.iloc[0]['PX_LAST'])
        lo, hi = spot * 0.97, spot * 1.03
        exp_limit = date.today() + timedelta(days=10)
        univ = bq.univ.options('SPX Index', include_expired=False, strike_range=(lo, hi))
        df_gex = _norm(bql.combined_df(bq.execute(
            'get(PX_LAST, OPEN_INT, DELTA, GAMMA) for(@universe)',
            {'@universe': univ}
        )).reset_index())
        date_col = next((c for c in df_gex.columns if 'DATE' in c.upper() or 'EXPIRE' in c.upper()), None)
        pc_col   = next((c for c in df_gex.columns if 'PUT_CALL' in c.upper()), None)
        stk_col  = next((c for c in df_gex.columns if 'STRIKE' in c.upper()), None)
        rows_gex = []
        for _, row in df_gex.iterrows():
            try:
                exp_dt = date.fromisoformat(str(row.get(date_col,''))[:10])
                if exp_dt > exp_limit: continue
            except Exception:
                continue
            oi = float(row.get('OPEN_INT') or 0)
            gm = float(row.get('GAMMA')    or 0)
            pc = str(row.get(pc_col, '')).upper()
            stk= float(row.get(stk_col) or 0)
            gex= oi * gm * spot**2 / 1e9 * (1 if pc=='CALL' else -1)
            rows_gex.append({'expiry': str(row.get(date_col,''))[:10], 'strike': round(stk,2),
                             'put_call': pc, 'open_int': int(oi), 'gamma': round(gm,6), 'gex_bn': round(gex,4)})
        _csv('gex_spx.csv', rows_gex, ['expiry','strike','put_call','open_int','gamma','gex_bn'])
        total = sum(r['gex_bn'] for r in rows_gex)
        calls = sum(r['gex_bn'] for r in rows_gex if r['put_call']=='CALL')
        puts  = sum(r['gex_bn'] for r in rows_gex if r['put_call']=='PUT')
        _csv('gex_summary.csv', [{
            'date': str(date.today()), 'spot': round(spot,2),
            'gex_total_bn': round(total,4), 'gex_call_bn': round(calls,4), 'gex_put_bn': round(puts,4),
            'direction': 'buy' if total>0.5 else ('sell' if total<-0.5 else 'flat'),
            'gamma_regime': 'long' if total>0 else ('short' if total<0 else 'flat'),
            'n_options': len(rows_gex),
        }], ['date','spot','gex_total_bn','gex_call_bn','gex_put_bn','direction','gamma_regime','n_options'])
    except Exception as e:
        print(f'  [ERRO] gex_spx: {e}')

    # ── 4. LETF Flows ────────────────────────────────────────────────────────
    try:
        LETFS = {
            'TQQQ US Equity':('TQQQ',3,'NDX'), 'SQQQ US Equity':('SQQQ',-3,'NDX'),
            'UPRO US Equity':('UPRO',3,'SPX'), 'SPXS US Equity':('SPXS',-3,'SPX'),
            'SOXL US Equity':('SOXL',3,'SOX'), 'SOXS US Equity':('SOXS',-3,'SOX'),
        }
        lstr = ', '.join(f'"{t}"' for t in LETFS)
        df_l = _norm(bql.combined_df(bq.execute(
            f'get(PX_LAST, FUND_TOTAL_ASSETS, FUND_NET_ASSET_VAL) for([{lstr}])'
        )).groupby(level=0).last())
        rows_l = []
        for bbg, (sym, lev, idx) in LETFS.items():
            r = df_l.loc[bbg] if bbg in df_l.index else {}
            aum = _v(r, 'FUND_TOTAL_ASSETS'); nav = _v(r, 'FUND_NET_ASSET_VAL') or _v(r, 'PX_LAST')
            rows_l.append({'ticker':sym,'leverage':lev,'index':idx,
                           'nav': round(nav,4) if nav else '',
                           'aum_b': round(aum/1e9,4) if aum else ''})
        _csv('letf_flows.csv', rows_l, ['ticker','leverage','index','nav','aum_b'])
    except Exception as e:
        print(f'  [ERRO] letf_flows: {e}')

    # ── Meta ─────────────────────────────────────────────────────────────────
    _csv('meta.csv', [{'generated_at': datetime.now().isoformat(), 'date': str(date.today())}],
         ['generated_at','date'])


# ── Loop principal ────────────────────────────────────────────────────────────
cycle = 0
while True:
    cycle += 1
    t0 = time.time()
    print(f'\n=== Ciclo {cycle} [{datetime.now().strftime("%H:%M:%S")}] ===')
    export_all()
    elapsed = time.time() - t0
    sleep = max(10, INTERVAL - elapsed)
    print(f'OK em {elapsed:.0f}s — próxima atualização em {sleep:.0f}s')
    time.sleep(sleep)